# Extraction Pipeline adapted to porous material surface coverage (coverage) extraction

End-to-end pipeline for extracting synthesis procedures **and** surface coverage (cov).
from (BET) adsorption and porous materials papers.

Adapted from tc_extrcation.ipynb

## Pipeline Overview

```
PDF (bytes)
    ↓
[0. PDF Extraction]        → Markdown text + embedded figures (base64)
    ↓
[1. Material Extraction]   → List of synthesized materials
    ↓
[2. Synthesis Extraction]  → GeneralSynthesisOntology per material
    ↓
[3. Tc Text Extraction]    → cov values mentioned in the paper text, per material  ★ NEW
    ↓
[4. Figure Extraction]     → Segmented subfigures (Florence-2)
    ↓
[5. Plot Data Extraction]  → (p, cov) data from cov(p) plots via Claude VLM
    ↓
[6. Plot Filtering]        → Keep only cov(p) plots (PlotFilterConfig.for_cov)
    ↓
[7. cov from VLM (Pipeline A)] → Direct cov extraction from cov(p) plots via Claude  ★ NEW
    ↓
[8. Link Series → Materials]
    ↓
[9. Aggregate & Save]
```

## Setup

In [ ]:
# ==============================================================================
# USER CONFIGURATION
# ==============================================================================

# Path to your porosity paper (PDF or markdown)
INPUT_PATH = "../../../data/mof_corpus/test_paper.pdf"  # or .md file with embedded images

# Output directory for results (inside the PDF papers folder)
PDF_DIR = r"./"
OUTPUT_DIR = f"{PDF_DIR}"

# Master CSV for multi-paper aggregation (appended after each run)
MASTER_CSV = f"{OUTPUT_DIR}/por_master.csv"

# Models
GEMINI_MODEL = (
    "gemini-2.5-flash-lite"  # For synthesis / porosity text extraction
)
CLAUDE_MODEL = (
    "claude-sonnet-4-6"  # For VLM plot data + cov extraction (better accuracy)
)
VLM_MODEL = CLAUDE_MODEL  # Switch to GEMINI_MODEL to use Gemini instead
LINKER_MODEL = "gemini-2.5-flash-lite"

# Set to True to skip figure extraction (synthesis + porosity text only)
SKIP_FIGURES = False

In [ ]:
# Load environment and imports
import json
import logging
import os
import re
import ssl
import sys
import warnings
from pathlib import Path

!{sys.executable} -m pip install transformers dspy 2>/dev/null || true
from dotenv import load_dotenv


# ── Fast dependency check (fail early, not at Step 4) ──
def _check_dependencies():
    """Verify critical dependencies are installed and compatible before running the pipeline."""
    errors = []

    # Check transformers + CLIP (needed for Florence-2 figure extraction)
    try:
        from transformers import CLIPImageProcessor
    except (ImportError, ModuleNotFoundError):
        try:
            import transformers

            ver = transformers.__version__
        except Exception:
            ver = "unknown"
        errors.append(
            f"transformers.CLIPImageProcessor not found (transformers=={ver}).\n"
            f"   Fix: pip install --upgrade transformers\n"
            f"   Or:  uv pip install --upgrade transformers"
        )

    # Check litellm (used for all LLM/VLM calls including Gemini)
    try:
        import litellm
    except ImportError:
        errors.append("litellm not installed. Fix: pip install litellm")

    # Check dspy (needed for text extraction)
    try:
        import dspy
    except ImportError:
        errors.append("dspy not installed. Fix: pip install dspy")

    if errors:
        print("=" * 60)
        print("DEPENDENCY CHECK FAILED")
        print("=" * 60)
        for e in errors:
            print(f"  ✗ {e}")
        print("=" * 60)
        raise ImportError(
            "Fix the above dependencies before running the pipeline."
        )
    else:
        print("[OK] Dependency check passed (transformers/CLIP, litellm, dspy)")


_check_dependencies()

# Fix SSL certificate issue for uv-managed Python on macOS
_ssl_cert = ssl.get_default_verify_paths().cafile
if _ssl_cert and os.path.exists(_ssl_cert):
    os.environ.setdefault("SSL_CERT_FILE", _ssl_cert)
    os.environ.setdefault("SSL_CERT_DIR", os.path.dirname(_ssl_cert))

src_path = Path("../../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

env_path = Path("../../.env")
load_dotenv(env_path, override=True)

warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

logging.getLogger("pydantic").setLevel(logging.ERROR)
logging.getLogger("LiteLLM").setLevel(logging.ERROR)
logging.getLogger("litellm").setLevel(logging.ERROR)

print("[OK] Environment loaded")
print(f"[OK] src path: {src_path}")
print(f"[OK] SSL_CERT_FILE: {os.environ.get('SSL_CERT_FILE', 'not set')}")

---
## Step 0: Load Paper Text

In [ ]:
from llm_synthesis.models.paper import Paper

# SI file detection helpers (same as synthesis_with_performance.ipynb)
SI_PATTERNS = [
    "_SI",
    "-SI",
    "_si",
    "-si",
    "_Supporting",
    "_supporting",
    "_Supplementary",
    "_supplementary",
    "_supp",
    "_Supp",
]


def find_si_file(main_paper_path: Path) -> Path | None:
    parent_dir = main_paper_path.parent
    main_stem = main_paper_path.stem
    for pattern in SI_PATTERNS:
        for ext in [".pdf", ".md", ".txt"]:
            si_path = parent_dir / f"{main_stem}{pattern}{ext}"
            if si_path.exists():
                return si_path
    return None


def load_file_text(path: Path, pdf_extractor=None) -> str:
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        if pdf_extractor is None:
            from llm_synthesis.transformers.pdf_extraction import (
                MistralPDFExtractor,
            )

            pdf_extractor = MistralPDFExtractor(structured=False)
        with open(path, "rb") as f:
            return pdf_extractor.forward(f.read())
    elif suffix in [".md", ".txt"]:
        with open(path, errors="replace") as f:
            return f.read()
    else:
        raise ValueError(f"Unsupported file type: {suffix}")


# Load main paper
input_path = Path(INPUT_PATH)
pdf_extractor = None

if input_path.suffix.lower() == ".pdf":
    print(f"Extracting text from PDF: {input_path.name}")
    from llm_synthesis.transformers.pdf_extraction import MistralPDFExtractor

    pdf_extractor = MistralPDFExtractor(structured=False)
    paper_text = load_file_text(input_path, pdf_extractor)
    print(f"   Main paper: {len(paper_text):,} characters")
elif input_path.suffix.lower() in [".md", ".txt"]:
    print(f"Loading markdown: {input_path.name}")
    paper_text = load_file_text(input_path)
    print(f"   Main paper: {len(paper_text):,} characters")
else:
    raise ValueError(f"Unsupported input type: {input_path}")

# Load SI file if exists
si_text = ""
si_path = find_si_file(input_path)
if si_path:
    print(f"   Found SI file: {si_path.name}")
    try:
        si_text = load_file_text(si_path, pdf_extractor)
        print(f"   SI text: {len(si_text):,} characters")
    except Exception as e:
        print(f"   [WARN] Failed to load SI file: {e}")

# Create Paper object
paper = Paper(
    name=input_path.stem,
    id=input_path.stem,
    publication_text=paper_text,
    si_text=si_text,
)

print(f"\n[OK] Paper loaded: {paper.name}")
print(f"   Main text: {len(paper.publication_text):,} chars")
print(f"   SI text: {len(paper.si_text):,} chars")

In [ ]:
import sys

print(sys.executable)

---
## Step 1: Extract Materials

In [ ]:
from llm_synthesis.transformers.material_extraction.dspy_extraction import (
    DspyTextExtractor,
    make_dspy_text_extractor_signature,
)
from llm_synthesis.utils.dspy_utils import get_llm_from_name
from llm_synthesis.utils.markdown_utils import clean_text

material_sig = make_dspy_text_extractor_signature(
    instructions=(
        "Extract ALL distinct porous material compositions that were synthesised "
        "and tested in this paper. IMPORTANT: If the paper studies multiple variants "
        "(e.g., different doping levels x=0.1, x=0.2, x=0.3, or different versions 1-MOF, 2-MOF etc.), list EACH variant "
        "as a separate material. Focus on materials that were actually synthesised, "
        "not just mentioned or referenced from other works."
    ),
    output_name="materials",
    output_description=(
        "ALL distinct synthesised material compositions as a comma-separated list "
        "using acronyms from the text. "
        "Never merge variants into a single generic name."
    ),
)

material_lm = get_llm_from_name(
    # "gemini-3.0-pro",
    # "gemini-3.0-flash",
    "gemini-2.5-flash-lite",
    model_kwargs={"temperature": 0.0, "max_tokens": 8000},
)
material_extractor = DspyTextExtractor(signature=material_sig, lm=material_lm)

print("Extracting materials...")
materials_text = material_extractor.forward(
    input=clean_text(paper.publication_text)
)

materials = [
    m.strip() for m in materials_text.replace("\n", ",").split(",") if m.strip()
]

print(f"\n{'=' * 60}")
print(f"MATERIALS FOUND ({len(materials)} total)")
print("=" * 60)
for i, mat in enumerate(materials, 1):
    print(f"  {i}. {mat}")

---
## Step 2: Extract Synthesis Procedures

In [ ]:
from llm_synthesis.metrics.judge.general_synthesis_judge import (
    DspyGeneralSynthesisJudge,
    make_general_synthesis_judge_signature,
)
from llm_synthesis.transformers.synthesis_extraction.dspy_synthesis_extraction import (
    DspySynthesisExtractor,
    make_dspy_synthesis_extractor_signature,
)

SYNTHESIS_SYSTEM_PROMPT = """You are a helpful assistant that extracts structured synthesis procedures from scientific papers.

IMPORTANT: For the synthesis_method field, you MUST choose from these exact values:
'PVD', 'CVD', 'arc discharge', 'ball milling', 'spray pyrolysis', 'electrospinning',
'sol-gel', 'hydrothermal', 'solvothermal', 'precipitation', 'coprecipitation', 'combustion',
'microwave-assisted', 'sonochemical', 'template-directed', 'solid-state', 'flux growth',
'float zone & Bridgman', 'arc melting & induction melting', 'spark plasma sintering',
'electrochemical deposition', 'chemical bath deposition', 'liquid-phase epitaxy', 'self-assembly',
'atomic layer deposition', 'molecular beam epitaxy', 'pulsed laser deposition', 'ion implantation',
'lithographic patterning', 'wet impregnation', 'incipient wetness impregnation', 'mechanical mixing',
'solution-based', 'mechanochemical', 'other'

For the target_compound_type field, you MUST choose from these exact values:
'framework & porous materials', 
'hybrid & organic-inorganic', 
'other'

If the exact method is not in the list, use the closest match or 'other'."""

synthesis_sig = make_dspy_synthesis_extractor_signature(
    instructions=(
        "Extract the complete structured synthesis procedure for the specified material. "
        "Include all steps, conditions (temperature, time, atmosphere), equipment, and precursors. "
        "Be thorough and preserve all quantitative details."
    ),
)

synthesis_lm = get_llm_from_name(
    GEMINI_MODEL,
    model_kwargs={"temperature": 0.0, "max_tokens": 32000, "num_retries": 3},
    system_prompt=SYNTHESIS_SYSTEM_PROMPT,
)
synthesis_extractor = DspySynthesisExtractor(
    signature=synthesis_sig, lm=synthesis_lm
)

# Judge
judge_lm = get_llm_from_name(
    GEMINI_MODEL,
    model_kwargs={"temperature": 0.1, "max_tokens": 8000},
)
judge_sig = make_general_synthesis_judge_signature()
judge = DspyGeneralSynthesisJudge(signature=judge_sig, lm=judge_lm)

print("[OK] Synthesis extractor and judge initialized")

In [ ]:
from llm_synthesis.models.paper import SynthesisEntry

all_syntheses = []
text_for_llm = clean_text(paper.publication_text)

for i, material in enumerate(materials, 1):
    print(f"\n{'=' * 60}")
    print(f"EXTRACTING SYNTHESIS {i}/{len(materials)}: {material}")
    print("=" * 60)

    try:
        synthesis = synthesis_extractor.forward(input=(text_for_llm, material))

        try:
            evaluation = judge.forward(
                (text_for_llm, json.dumps(synthesis.model_dump()), material)
            )
            print(f"   [OK] Score: {evaluation.scores.overall_score}/5.0")
        except Exception as e:
            print(f"   [WARN] Judge failed: {e}")
            evaluation = None

        all_syntheses.append(
            SynthesisEntry(
                material=material,
                synthesis=synthesis,
                evaluation=evaluation,
            )
        )
        print(f"   Method: {synthesis.synthesis_method}")
        print(f"   Steps: {len(synthesis.steps)}")

    except Exception as e:
        print(f"   [ERROR] {e}")
        all_syntheses.append(
            SynthesisEntry(
                material=material,
                synthesis=None,
                evaluation=None,
            )
        )

print(f"\n[OK] Extracted synthesis for {len(all_syntheses)} materials")

---
## Step 3: Extract Adsorption data from Text ★ NEW

Use an LLM to extract critical temperature (Tc) values mentioned in the paper text.
This gives us the **text-reported** Tc for comparison with the VLM-extracted Tc from plots.

In [ ]:
# Create a porosity/ BET adsorption text extractor using the same DspyTextExtractor pattern
cov_text_sig = make_dspy_text_extractor_signature(
    signature_name="TextToCoverage",
    instructions=(
        "Extract ALL BET surface area values, also known as surface coverage, reported in this porous materials paper. "
        "For EACH material that has a BET surface area, or coverage, value mentioned in the text, report:\n"
        "  - The material formula\n"
        "  - coverage, surface area, or loading: the maximum loading of a species, usually gas, per gram of material, or when the plot starts to plateau.\n"
        "  - p_cov (if explicitly reported): pressure where the maximum loading is achieved.\n"
        "  - Whether the material is porous (YES/NO)\n\n"
        "IMPORTANT RULES:\n"
        "1. Only extract values explicitly stated in the text. Do NOT estimate or calculate.\n"
        "2. Most papers report only a SINGLE surface area coverage value without distinguishing onset/mid/zero. "
        "   In that case, report it as 'coverage: <value> m^2/g' and use 'NR' for p_cov.\n"
        "3. Only report p_cov if the paper EXPLICITLY states it as separate value.\n"
        "4. If a material is mentioned but no coverage is given, report 'porosity: NO' or 'coverage: NR'.\n"
        "5. Keep your reasoning SHORT. Focus on extracting values, not explaining the paper.\n"
        "6. If the pressure is given as p/p0 or with respect to STP (standard temperature and pressure), leave the units as 'p0 atm'."
    ),
    input_description="The full publication text from a porous material paper.",
    output_name="coverage_values",
    output_description=(
        "For each material, one line in the format:\n"
        "material_formula | porous: YES/NO | coverage: <value> m^2/g | p_cov: <value> atm\n"
        "Use 'NR' (not reported) for values not explicitly stated in the text.\n"
        "Examples:\n"
        "  Ba0.6K0.4Fe2As2 | porous: YES | coverage: 36 m^2/g | p_cov: 34 p0 atm\n"
        "  MgB2 | porous: YES | coverage: 1000 m^2/g | p_cov: NR\n"
        "  BaFe2As2 | porous: NO | coverage: 2097 m^2/g | p_cov: NR"
    ),
)

cov_text_lm = get_llm_from_name(
    GEMINI_MODEL,
    model_kwargs={"temperature": 0.0, "max_tokens": 16384},
)
cov_text_extractor = DspyTextExtractor(signature=cov_text_sig, lm=cov_text_lm)

# Truncate very long papers to avoid output truncation issues
# DSPy wraps the text in JSON and adds reasoning — keep input reasonable
MAX_TEXT_CHARS = 60_000
if len(text_for_llm) > MAX_TEXT_CHARS:
    print(
        f"[INFO] Paper text is {len(text_for_llm):,} chars — truncating to {MAX_TEXT_CHARS:,} for porosity text extraction"
    )
    cov_input_text = text_for_llm[:MAX_TEXT_CHARS]
else:
    cov_input_text = text_for_llm

print("Extracting coverage values from text...")
try:
    cov_text_raw = cov_text_extractor.forward(input=cov_input_text)
except Exception as e:
    print(f"[WARN] Coverage text extraction failed: {e}")
    print("[WARN] Falling back to empty Coverage text results")
    cov_text_raw = ""

print(f"\n{'=' * 60}")
print("COVERAGE VALUES FROM TEXT")
print("=" * 60)
print(cov_text_raw)

In [ ]:
# Parse the cov (i.e. where the adsorption isotherm plateaus) text output into a structured dict
def parse_cov_text_response(raw_text: str) -> dict:
    """
    Parse the por text extraction response into a dict:
    {material_name: {"porous": bool, "coverage": float|None, "p_cov": float|None}}
    """
    results = {}
    for line in raw_text.strip().split("\n"):
        line = line.strip()
        if not line or "|" not in line:
            continue
        parts = [p.strip() for p in line.split("|")]
        if len(parts) < 2:
            continue

        material = parts[0].strip()
        entry = {"porous": False, "coverage": None, "p_cov": None}

        for part in parts[1:]:
            part_lower = part.lower().strip()
            if "porous" in part_lower:
                entry["porous"] = "yes" in part_lower
            else:
                # Match "coverage: <value> <any unit>" or "p_cov: <value> <any unit>"
                match = re.match(
                    r"(coverage|p_cov)\s*:\s*(\d+\.?\d*)",
                    part_lower,
                )
                if match:
                    key = match.group(1)
                    val = float(match.group(2))
                    entry[key] = val

        results[material] = entry

    return results


cov_from_text = parse_cov_text_response(cov_text_raw)

print(f"\nParsed coverage values for {len(cov_from_text)} materials:")
for mat, vals in cov_from_text.items():
    sc = "YES" if vals["porous"] else "NO"
    cov = vals["coverage"]
    cov_str = f"{cov:.1f} m^2/g" if cov is not None else "NR"
    print(f"  {mat}: porous={sc}, cov={cov_str}")

---
## Step 4: Extract Figures

In [ ]:
if SKIP_FIGURES:
    print("[SKIP] Skipping figure extraction")
    figures = []
else:
    from llm_synthesis.transformers.figure_extraction import (
        FigureExtractorMarkdown,
    )

    extractor = FigureExtractorMarkdown(
        segmenter="florence",
        florence_repo_id="amayuelas/plot-visualization-florence-2-lora-32",
    )
    print("Extracting figures using Florence-2...")
    figures = extractor.forward(paper.publication_text)

    print(f"\n{'=' * 60}")
    print(f"FIGURES FOUND ({len(figures)} subfigures)")
    print("=" * 60)
    for i, fig in enumerate(figures):
        print(
            f"  {i + 1}. {fig.figure_reference or f'Figure {i}'}: {fig.figure_class}"
        )

---
## Step 5: Extract Plot Data (Claude VLM)

Send all figures to Claude to extract (T, R) coordinate data.

In [ ]:
if SKIP_FIGURES or not figures:
    print("[SKIP] Skipping plot data extraction")
    plots = []
    plot_figures = []
else:
    from llm_synthesis.models.figure import FigureInfoWithPaper
    from llm_synthesis.utils.figure_utils import clean_text_from_images

    _is_claude = VLM_MODEL.startswith("claude")

    if _is_claude:
        from llm_synthesis.transformers.plot_extraction.claude_extraction.plot_data_extraction import (
            ClaudeLinePlotDataExtractor,
        )

        plot_extractor = ClaudeLinePlotDataExtractor(
            model_name=VLM_MODEL, max_tokens=4096
        )
        print(
            f"Extracting data from {len(figures)} figures using Claude VLM ({VLM_MODEL})..."
        )
    else:
        from llm_synthesis.transformers.plot_extraction.litellm_plot_data_extraction import (
            LiteLLMPlotDataExtractor,
        )

        plot_extractor = LiteLLMPlotDataExtractor(
            model=f"gemini/{VLM_MODEL}", max_tokens=4096
        )
        print(
            f"Extracting data from {len(figures)} figures using Gemini VLM ({VLM_MODEL})..."
        )

    plots = []
    plot_figures = []

    for i, fig in enumerate(figures):
        print(
            f"\n  [{i + 1}/{len(figures)}] {fig.figure_reference or f'Figure {i}'} ({fig.figure_class})"
        )

        fig_with_paper = FigureInfoWithPaper(
            base64_data=fig.base64_data,
            alt_text=fig.alt_text,
            position=fig.position,
            context_before=fig.context_before,
            context_after=fig.context_after,
            figure_reference=fig.figure_reference,
            figure_class=fig.figure_class,
            quantitative=fig.quantitative,
            paper_text=clean_text_from_images(paper.publication_text),
            si_text=paper.si_text,
        )

        try:
            plot_data = plot_extractor.forward(fig_with_paper)
            if plot_data and plot_data.name_to_coordinates:
                plots.append(plot_data)
                plot_figures.append(fig)
                series_names = list(plot_data.name_to_coordinates.keys())
                print(f"    [OK] {len(series_names)} series: {series_names}")
                print(
                    f"    Axes: x={plot_data.x_axis_label!r} [{plot_data.x_axis_unit!r}]"
                    f"  y={plot_data.y_left_axis_label!r} [{plot_data.y_left_axis_unit!r}]"
                )
            else:
                print("    [--] No extractable data")
        except Exception as e:
            print(f"    [ERROR] {e}")

    print(f"\n[OK] Extracted data from {len(plots)} plots")
    print(f"   VLM cost: ${plot_extractor.get_cost():.4f}")

---
## Step 6: Filter for cov(p) Plots

Use `PlotFilterConfig.for_coverage()` to keep only plots that look like
coverage/ loading vs pressure (cov(p)) curves.

In [ ]:
from llm_synthesis.config.plot_filter_config import PlotFilterConfig
from llm_synthesis.transformers.performance_linking.plot_filter import (
    PlotFilter,
)

filter_config = PlotFilterConfig.for_coverage()
plot_filter = PlotFilter(filter_config)

print("Plot Filter (Coverage):")
print(f"   X-axis labels: {filter_config.x_axis_labels}")
print(f"   X-axis units: {filter_config.x_axis_units}")
print(
    f"   Y-axis keywords: {filter_config.y_axis_keywords[:5]}... ({len(filter_config.y_axis_keywords)} total)"
)
print(
    f"   Y-axis units: {filter_config.y_axis_units[:5]}... ({len(filter_config.y_axis_units)} total)"
)
print(f"   Y-axis exclude: {filter_config.y_axis_exclude_patterns[:4]}...")


def fallback_check_rt_plot(plot, fig) -> bool:
    """Fallback: check if a plot with missing axis metadata is a cov(p) plot
    by looking at the figure caption/context and the data itself."""
    context = f"{fig.context_before or ''} {fig.context_after or ''} {fig.alt_text or ''}".lower()
    cov_p_context_hints = [
        "coverage",
        "loading",
        "surface area",
        "molecules adsorbed",
        "cm^3/g",
        "surface coverage",
        "adsorption isotherm",
        "nitrogen loading",
        "pressure dependence of the porosity",
        "BET curve",
        "BET surface area",
        "p/ p0",
    ]  # Add more as needed
    return any(hint in context for hint in cov_p_context_hints)


def _is_axis_missing(label, unit) -> bool:
    """Check if axis metadata is effectively missing (None or empty string)."""
    return (not label or not label.strip()) and (not unit or not unit.strip())


if plots:
    # ── Debug: show what Claude VLM extracted for each plot ──
    print(f"\n{'=' * 60}")
    print("DEBUG: Extracted axis labels for all plots")
    print("=" * 60)
    for i, plot in enumerate(plots):
        fig = plot_figures[i]
        print(f"\n  Plot {i}: {plot.title or 'N/A'}")
        print(f"    x_axis_label: {plot.x_axis_label!r}")
        print(f"    x_axis_unit:  {plot.x_axis_unit!r}")
        print(f"    y_left_axis_label: {plot.y_left_axis_label!r}")
        print(
            f"    y_left_axis_unit:  {plot.y_left_axis_unit!r}"
        )  # for sovergae vs pressure plots it's common to include the pressure units on the y-axis as well, e.g. "coverage (m^2/g) at 1 atm" pressure as p/ p0 on the x axis

        x_ok = filter_config.is_relevant_x_axis(
            plot.x_axis_label, plot.x_axis_unit
        )
        y_ok = filter_config.is_relevant_y_axis(
            plot.y_left_axis_label, plot.y_left_axis_unit
        )
        print(f"    -> x_axis relevant: {x_ok}")
        print(f"    -> y_axis relevant: {y_ok}")

        if not x_ok or not y_ok:
            fb = fallback_check_rt_plot(plot, fig)
            print(f"    -> fallback (context check): {fb}")

    # First pass: standard filter
    relevant_plots, skip_counts = plot_filter.filter_plots(
        plots, log_skipped=False
    )

    # Second pass: for rejected plots, try fallback if axis metadata is missing/empty
    rejected_indices = {i for i in range(len(plots))} - {
        idx for idx, _ in relevant_plots
    }
    for i in sorted(rejected_indices):
        plot = plots[i]
        fig = plot_figures[i]
        # Use fallback if EITHER axis metadata is missing/empty (not if present but wrong type)
        x_missing = _is_axis_missing(plot.x_axis_label, plot.x_axis_unit)
        y_missing = _is_axis_missing(
            plot.y_left_axis_label, plot.y_left_axis_unit
        )
        if (x_missing or y_missing) and fallback_check_rt_plot(plot, fig):
            relevant_plots.append((i, plot))
            missing_info = []
            if x_missing:
                missing_info.append("x-axis")
            if y_missing:
                missing_info.append("y-axis")
            print(
                f"\n  [FALLBACK] Plot {i} included via context check (missing {', '.join(missing_info)} metadata)"
            )

    # Sort by index
    relevant_plots.sort(key=lambda x: x[0])

    print(f"\n{'=' * 60}")
    print("PLOT FILTERING RESULTS")
    print("=" * 60)
    print(f"  Total plots: {len(plots)}")
    print(f"  cov(p) plots: {len(relevant_plots)}")
    print(f"  Skipped (wrong x-axis): {skip_counts.get('not_relevant_x', 0)}")
    print(f"  Skipped (wrong y-axis): {skip_counts.get('not_relevant_y', 0)}")

    for idx, plot in relevant_plots:
        print(f"\n  Plot {idx}: {plot.title or 'N/A'}")
        print(f"    X: {plot.x_axis_label} [{plot.x_axis_unit}]")
        print(f"    Y: {plot.y_left_axis_label} [{plot.y_left_axis_unit}]")
        print(f"    Series: {list(plot.name_to_coordinates.keys())}")
else:
    print("\n[SKIP] No plots to filter")
    relevant_plots = []

---
## Step 7: Extract maximum surface coverage (cov) from cov(p) Plots via VLM (Pipeline A) ★ NEW

For each relevant cov(p) plot, ask Claude to directly determine cov using
the geometric construction (cov_onset, cov_final, cov_mid) — this is the approach
that was validated with ~11% average error for the Tc script.

!!!!!! might need to get adapted -> I did my best but this is still building on top of the Tc work !!!!!

In [ ]:
# The validated cov extraction prompt (Pipeline A from batch_tc_extraction.py)
# Improved with Kondo/metallic behavior warning and transition width sanity check
# Uses {series_name_instruction} placeholder — filled per-plot in cell-22
DIRECT_COV_PROMPT_TEMPLATE = """
You are analyzing a surface coverage (or loading) vs pressure plot of an adsorption isotherm from a
porous material paper. Your task is to determine the maximum surface coverage (or loading)
for each series using the standard geometric construction.


STEP 0 — EXAMINE THE FULL FIGURE (main plot + insets/panels):

a) Identify ALL panels in the figure:
- Main isotherm (e.g., uptake vs pressure, loading vs p/p0)
- Insets (zoomed regions, BET plots, pore size distributions, etc.)

For each panel, describe:
- What quantity is on each axis (e.g., loading [mmol/g] vs pressure [bar] or relative pressure p/p0)
- Axis ranges and tick marks
- Whether it contains information relevant to:
  - Maximum uptake (saturation capacity)
  - Low-pressure uptake (Henry's law region)
  - Step transitions (e.g., gate-opening, capillary condensation)

b) CATEGORIZE each panel into one of these types:

(i) ZOOMED ISOTHERM REGION
- Same variables as main plot but narrower pressure range (often low p/p0)
- Use for accurate reading of initial uptake or steep transitions

(ii) SUMMARY / DERIVED DATA
- BET plots, Langmuir fits, uptake vs temperature, working capacity plots
- Use directly if they report:
  - Surface area
  - Saturation loading
  - Uptake at specific conditions

(iii) OTHER
- PSD, structure, simulations, etc.
- Note but do not use for numerical extraction

c) Read ALL numbered tick marks on the main plot axes.
If a zoomed inset exists, read its tick marks separately.

d) CRITICAL — SCALE AWARENESS:
- Check if pressure axis is linear or logarithmic
- If pressure spans multiple orders of magnitude, be careful with interpolation
- If most uptake change occurs in a narrow region, prefer zoomed inset
- Avoid reading values from visually compressed regions

STEP 1 — IDENTIFY SERIES:
{series_name_instruction}
List every distinct isotherm (e.g., different materials, temperatures, gases).

STEP 2 — READ KEY VALUES FOR EACH SERIES:

For EACH series, extract:

a) Uptake_at_lowest_pressure
- Value at the smallest pressure shown
- Important for characterizing Henry's law region

b) Uptake_at_highest_pressure
- Value at the highest pressure (approximate saturation loading)

c) Pressure_range
- Minimum and maximum pressure explicitly shown


STEP 3 — IDENTIFY ISOTHERM TYPE / FEATURES:

Classify qualitatively (if possible):
- Type I (microporous, rapid saturation)
- Type II / IV (mesoporous, possible hysteresis)
- Stepped isotherm (framework flexibility, gate opening)

Check for:
- Plateau → indicates saturation region
- Step(s) → indicates structural transition or pore filling
- Hysteresis → difference between adsorption and desorption


STEP 4 — DETERMINE MAXIMUM UPTAKE (SATURATION CAPACITY):

For each series:

a) Identify plateau region:
- Region where uptake stops increasing significantly with pressure

b) Define:
- Uptake_max = value in plateau at highest pressure

c) If no clear plateau:
- Report uptake at highest measured pressure
- Add note: "no clear saturation"

d) If summary/inset provides uptake directly:
- Prefer inset value (higher precision)
STEP 5 — IDENTIFY CHARACTERISTIC PRESSURES (OPTIONAL):

If applicable:

a) Step pressure (P_step):
- Pressure at which a sharp increase in uptake occurs

b) Knee pressure (P_knee):
- Transition from steep uptake to plateau

c) Low-pressure regime:
- Region where uptake is approximately linear with pressure

STEP 6 — SANITY CHECKS:

- Uptake should increase monotonically with pressure (adsorption branch)
- Sudden drops may indicate:
  - Desorption branch
  - Misidentified curve

- Cross-check trends:
  - Higher surface area → higher saturation uptake
  - Stronger adsorption → higher low-pressure uptake

STEP 7 — RELATIVE COMPARISON:

For multiple series:

- Identify highest Uptake_max
- Identify strongest low-pressure uptake
- Identify lowest P_step (strongest affinity)

Avoid reporting identical values unless clearly indistinguishable.

KEY CONCEPTUAL NOTE:

Adsorption isotherms are continuous curves, not sharp transitions.

Therefore, extract:
- Regime-dependent quantities (low-pressure uptake, plateau, steps)
rather than a single characteristic point.

Output format:

inset_detected: <yes/no>
inset_type: <"zoomed_isotherm" | "summary_data" | "other" | "none">
inset_description: <brief description of what the inset shows, or "N/A">
inset_axes: <tick marks of inset if present, otherwise "N/A">
inset_values: <series: value (with units, e.g. p0 or bar), series: value p0, ... if summary_data type, otherwise "N/A">
main_x_axis_ticks: <list>
main_y_axis_ticks: <list>

Series: <name>
uptake_at_lowest_pressure: <value and unit>
uptake_at_highest_pressure: <value and unit>
pressure_range: <min–max with units>

isotherm_type: <Type I / Type II / Type IV / stepped / unknown>
has_plateau: <YES/NO>
has_step: <YES/NO>
has_hysteresis: <YES/NO>

[If has_plateau = YES:]
uptake_max: <value and unit>
source: <"inset" or "main plot" or "zoomed inset">

[If has_plateau = NO:]
uptake_max: <value and unit>
note: "no clear saturation"
source: <"inset" or "main plot" or "zoomed inset">

[If has_step = YES:]
P_step: <value and unit>

[If identifiable:]
P_knee: <value and unit>

relative_comparison:
- highest_uptake_max: <series name>
- strongest_low_pressure_uptake: <series name>
- lowest_P_step: <series name or "N/A">

Do not output any other text.
"""


def build_cov_prompt(known_series_names: list[str] | None = None) -> str:
    """Build the max coverage extraction prompt, optionally injecting known series names
    from Step 5 so the VLM uses the EXACT same names."""
    if known_series_names and len(known_series_names) > 0:
        names_list = "\n".join(
            f"  {i + 1}. {name}" for i, name in enumerate(known_series_names)
        )
        instruction = (
            "The following series were previously identified in this plot. "
            "You MUST use these EXACT names in your output (in the 'Series:' lines):\n"
            f"{names_list}\n\n"
            "If you see additional curves not in this list, add them with descriptive names."
        )
    else:
        instruction = (
            "List every distinct curve (by legend label, color, marker)."
        )
    return DIRECT_COV_PROMPT_TEMPLATE.format(
        series_name_instruction=instruction
    )


def parse_direct_cov_response(response_text: str) -> dict:
    """Parse the VLM max coverage response into {series_name: {porous, p_step, ...}}.

    Also parses inset_cov_values from the header block (before any Series: lines)
    and attaches them to the corresponding series as 'inset_cov'.
    """
    results = {}
    current = None
    inset_cov_raw = None  # raw string from "inset_cov_values:" line

    for line in response_text.strip().split("\n"):
        line = line.strip()

        # Parse header-level inset_cov_values (before any Series: block)
        if line.lower().startswith("inset_cov_values:") and current is None:
            inset_cov_raw = line.split(":", 1)[1].strip()
            continue

        if line.startswith("Series:"):
            current = line.split(":", 1)[1].strip()
            results[current] = {}
        elif current and ":" in line:
            key, val = line.split(":", 1)
            key = key.strip().lower().replace(" ", "_")
            val = val.strip()
            if key == "porous":
                results[current][key] = val.upper().startswith("YES")
            elif key in ("reason", "source"):
                results[current][key] = val
            else:
                match = re.search(r"(\d+\.?\d*|\d*\.\d+)", val)
                if match:
                    try:
                        results[current][key] = float(match.group(1))
                    except ValueError:
                        results[current][key] = val
                elif "no transition" in val.lower() or "n/a" in val.lower():
                    results[current][key] = None
                else:
                    results[current][key] = val

    # Parse inset_cov_values and attach to matching series
    # Format: "x=0.0: 38 K, x=0.10: 33 K, ..." or "N/A"
    if inset_cov_raw and inset_cov_raw.lower() not in ("n/a", "none", ""):
        inset_pairs = _parse_inset_cov_values(inset_cov_raw, results)
        for series_name, cov_val in inset_pairs.items():
            if series_name in results:
                results[series_name]["inset_cov"] = cov_val

    return results


def _parse_inset_cov_values(raw: str, known_series: dict) -> dict:
    """Parse the inset_cov_values string into {series_name: float}.

    Handles formats like:
      "x=0.0: 38 p/p0, x=0.10: 33 p/p0, x=0.20: 29 p/p0"
      "x=0.0: 38 bar, x=0.10: 33 bar, x=0.20: 29 bar"

    Fuzzy-matches against known_series keys.
    """
    result = {}
    # Split by comma to get individual entries
    for entry in raw.split(","):
        entry = entry.strip()
        if not entry or ":" not in entry:
            continue
        name_part, val_part = entry.rsplit(":", 1)
        name_part = name_part.strip()
        val_part = val_part.strip()
        # Extract numeric value
        match = re.search(r"(\d+\.?\d*|\d*\.\d+)", val_part)
        if match:
            tc_val = float(match.group(1))
            # Try exact match first
            if name_part in known_series:
                result[name_part] = tc_val
            else:
                # Fuzzy match: normalize and compare
                np_lower = name_part.lower().replace(" ", "")
                for ks in known_series:
                    ks_lower = ks.lower().replace(" ", "")
                    if (
                        np_lower == ks_lower
                        or np_lower in ks_lower
                        or ks_lower in np_lower
                    ):
                        result[ks] = tc_val
                        break
    return result


print(
    "[OK] Cov VLM extractor ready (with series name injection + general inset handling)"
)

In [ ]:
# ── Sanity check: flag suspiciously wide "plateaus" + inset cross-validation ──
def sanity_check_delta_coverage(vlm_results: dict) -> dict:
    corrected = {}
    for series_name, vals in vlm_results.items():
        entry = dict(vals)

        # Normalize: VLM returns uptake_max — map to coverage_mid for downstream
        if "uptake_max" in entry and "coverage_mid" not in entry:
            entry["coverage_mid"] = entry["uptake_max"]

        # If porous not set, infer from has_plateau / uptake_max
        if "porous" not in entry:
            has_plateau = str(entry.get("has_plateau", "")).upper()
            entry["porous"] = (
                has_plateau == "YES" or entry.get("coverage_mid") is not None
            )

        vlm_says_porous = entry.get("porous", False)
        if not vlm_says_porous:
            inset_cov = entry.get("inset_coverage")
            if inset_cov is not None and inset_cov > 0:
                entry["porous"] = True
                entry["coverage_mid"] = inset_cov
                entry["source"] = "inset"
                entry["_inset_override"] = (
                    f"VLM said porous=NO, but inset coverage={inset_cov:.1f} m^2/g found."
                )
            corrected[series_name] = entry
            continue

        # Cross-validate against inset if available
        cov_mid = entry.get("coverage_mid")
        inset_cov = entry.get("inset_coverage")
        if cov_mid is not None and inset_cov is not None and inset_cov > 0:
            relative_diff = abs(cov_mid - inset_cov) / inset_cov
            if relative_diff > 0.20:
                entry["coverage_mid"] = inset_cov
                entry["source"] = "inset"
                entry["_inset_override"] = (
                    f"coverage_mid={cov_mid:.1f} differs from inset={inset_cov:.1f} "
                    f"by {relative_diff * 100:.0f}% (>20%). Using inset."
                )

        corrected[series_name] = entry
    return corrected


# Run coverage extraction on each relevant isotherm plot
import litellm

_is_claude_vlm = VLM_MODEL.startswith("claude")
if _is_claude_vlm:
    import anthropic

    _anthropic_client = anthropic.Anthropic()

cov_from_vlm = {}
_vlm_total_cost = 0.0


def _img_type(b64: str) -> str:
    return "image/jpeg" if b64.startswith("/9j/") else "image/png"


def _call_coverage_vlm(fig_b64: str, prompt: str) -> tuple[str, float]:
    if _is_claude_vlm:
        resp = _anthropic_client.messages.create(
            model=VLM_MODEL,
            max_tokens=4096,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": _img_type(fig_b64),
                                "data": fig_b64,
                            },
                        },
                        {"type": "text", "text": prompt},
                    ],
                }
            ],
        )
        text = resp.content[0].text
        cost = (resp.usage.input_tokens * 3e-6) + (
            resp.usage.output_tokens * 15e-6
        )
        return text, cost
    else:
        image_type = _img_type(fig_b64).split("/")[1]
        resp = litellm.completion(
            model=f"gemini/{VLM_MODEL}",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/{image_type};base64,{fig_b64}"
                            },
                        },
                        {"type": "text", "text": prompt},
                    ],
                }
            ],
            max_tokens=4096,
            temperature=0.0,
        )
        text = resp.choices[0].message.content
        cost = litellm.completion_cost(resp) or 0.0
        return text, cost


if relevant_plots:
    for idx, plot in relevant_plots:
        fig = plot_figures[idx]
        known_series = list(plot.name_to_coordinates.keys())
        prompt = build_cov_prompt(known_series_names=known_series)

        print(f"\n{'=' * 60}")
        print(
            f"COVERAGE EXTRACTION — Plot {idx}: {fig.figure_reference or 'N/A'}"
        )
        print(f"  X: {plot.x_axis_label} [{plot.x_axis_unit}]")
        print(f"  Y: {plot.y_left_axis_label} [{plot.y_left_axis_unit}]")
        print(f"  Known series from Step 5: {known_series}")
        print("=" * 60)

        try:
            response, cost = _call_coverage_vlm(fig.base64_data, prompt)
            _vlm_total_cost += cost

            parsed = parse_direct_cov_response(response)
            corrected = sanity_check_delta_coverage(parsed)
            cov_from_vlm[idx] = corrected

            for series_name, vals in corrected.items():
                porous = "YES" if vals.get("porous") else "NO"
                coverage_mid = vals.get("coverage_mid")
                cov_str = f"{coverage_mid:.1f} cm³/g" if coverage_mid else "N/A"
                source = vals.get("source", "main plot")
                override = vals.get(
                    "_sanity_override", vals.get("_inset_override", "")
                )
                suffix = f"  ⚠ {override}" if override else ""
                print(
                    f"  {series_name}: porous={porous}, uptake_max={cov_str} (source: {source}){suffix}"
                )

        except Exception as e:
            print(f"  [ERROR] {e}")

    print(f"\n[OK] Coverage VLM cost: ${_vlm_total_cost:.4f}")
else:
    print("[SKIP] No isotherm plots found for coverage extraction")

---
## Step 8: Link Plot Series to Materials

In [ ]:
if SKIP_FIGURES or not relevant_plots:
    print("[SKIP] Skipping performance linking")
    plot_mappings = []
else:
    from llm_synthesis.models.performance import PlotMaterialMapping
    from llm_synthesis.transformers.performance_linking.base import LinkingInput
    from llm_synthesis.transformers.performance_linking.series_material_linker import (
        SeriesMaterialLinker,
    )

    print(f"Linking plot series to {len(materials)} materials...")

    linker_lm = get_llm_from_name(
        LINKER_MODEL,
        model_kwargs={"temperature": 0.0, "max_tokens": 8000},
    )
    series_linker = SeriesMaterialLinker(lm=linker_lm)

    plot_mappings = []

    for idx, plot in relevant_plots:
        fig = plot_figures[idx]
        series_names = list(plot.name_to_coordinates.keys())

        print(f"\n  Plot {idx}: {len(series_names)} series")
        print(f"    Series: {series_names}")

        context = f"{fig.context_before} {fig.context_after}"
        plot_meta = {
            "title": plot.title,
            "x_axis_label": plot.x_axis_label,
            "x_axis_unit": plot.x_axis_unit,
            "y_left_axis_label": plot.y_left_axis_label,
            "y_left_axis_unit": plot.y_left_axis_unit,
        }

        linking_input = LinkingInput(
            materials=materials,
            series_names=series_names,
            context=context,
            plot_metadata=plot_meta,
        )
        validated_mappings = series_linker.forward(linking_input)

        matched_series = {m.series_name for m in validated_mappings}
        unmatched = [s for s in series_names if s not in matched_series]

        plot_mappings.append(
            PlotMaterialMapping(
                plot_index=idx,
                figure_reference=fig.figure_reference,
                mappings=validated_mappings,
                unmatched_series=unmatched,
            )
        )

        for m in validated_mappings:
            print(
                f"    '{m.series_name}' -> '{m.material_name}' ({m.confidence})"
            )
        if unmatched:
            print(f"    [WARN] Unmatched: {unmatched}")

    print("\n[OK] Linking complete")

---
## Step 9: Aggregate & Build Final Results

Combine everything: synthesis + Tc from text + Tc from VLM + R(T) data per material.

In [ ]:
from llm_synthesis.utils.performance_utils import (
    aggregate_all_materials_performance,
)

# ── Fuzzy matching helpers ──


def _normalize_formula(s: str) -> str:
    """Normalize a material formula for fuzzy matching.
    Strips parenthetical suffixes like '(9hr-reduced)', normalizes unicode
    (δ→delta, subscripts→digits), lowercases, removes whitespace."""
    # Strip parenthetical condition suffixes like "(9hr-reduced)", "(as-sintered)"
    base = re.sub(r"\s*\([^)]*\)\s*$", "", s).strip()
    # Normalize unicode: δ→delta, subscript digits, etc.
    base = base.replace("δ", "delta").replace("Δ", "delta")
    base = (
        base.replace("₀", "0")
        .replace("₁", "1")
        .replace("₂", "2")
        .replace("₃", "3")
    )
    base = (
        base.replace("₄", "4")
        .replace("₅", "5")
        .replace("₆", "6")
        .replace("₇", "7")
    )
    base = base.replace("₈", "8").replace("₉", "9").replace("₋", "-")
    return base.lower().replace(" ", "").replace("−", "-")


def _find_matching_text_coverage(material: str, cov_from_text: dict) -> list:
    """Find ALL cov_from_text entries matching `material`.

    Priority:
    1. Exact normalized match (IRMOF-1 == IRMOF-1)
    2. Prefix match only if no other material in the list shares the same prefix
       (e.g. "IRMOF" matches "IRMOF-1" only if no "IRMOF-1" exact entry exists)
    """
    mat_norm = _normalize_formula(material)
    exact = []
    prefix = []
    for cov_key, cov_entry in cov_from_text.items():
        key_norm = _normalize_formula(cov_key)
        if key_norm == mat_norm:
            exact.append((cov_key, cov_entry))
        elif mat_norm.startswith(key_norm + "-") or (
            key_norm
            and mat_norm.startswith(key_norm)
            and len(key_norm) < len(mat_norm)
        ):
            prefix.append((cov_key, cov_entry))
    # Use exact matches if available; fall back to prefix only when no exact match
    matches = exact if exact else prefix
    matches.sort(
        key=lambda x: (
            not x[1].get("porous", False),
            -(x[1].get("coverage") or 0),
        )
    )
    return matches


def _find_vlm_coverage_for_series(series_name: str, vlm_results: dict) -> dict:
    """Fuzzy-match a series name from Step 5/8 against VLM coverage keys from Step 7.
    The VLM in Step 7 should now use the same names (via prompt injection),
    but this provides fallback matching just in case."""
    # Exact match first
    if series_name in vlm_results:
        return vlm_results[series_name]

    # Normalize both and try
    def _norm(s):
        s = (
            s.replace("₂", "2")
            .replace("₄", "4")
            .replace("₇", "7")
            .replace("₁", "1")
        )
        s = (
            s.replace("₃", "3")
            .replace("₅", "5")
            .replace("₆", "6")
            .replace("₈", "8")
        )
        s = (
            s.replace("₉", "9")
            .replace("₀", "0")
            .replace("₋", "-")
            .replace("δ", "delta")
        )
        return s.lower().replace(" ", "")

    sn = _norm(series_name)
    for vlm_key, vlm_val in vlm_results.items():
        # Check if one contains the other (e.g., "9hr-reduced Pr₂Ba₄..." contains "9hr-reduced")
        vk = _norm(vlm_key)
        if sn in vk or vk in sn:
            return vlm_val
        # Check if they share a distinguishing prefix (e.g., "9hr-reduced")
        sn_parts = re.split(r"[\s_-]+", series_name.lower())
        vk_parts = re.split(r"[\s_-]+", vlm_key.lower())
        # If first 1-2 tokens match, it's likely the same series
        if (
            len(sn_parts) >= 1
            and len(vk_parts) >= 1
            and sn_parts[0] == vk_parts[0]
        ):
            return vlm_val
    # Single-series fallback: if the VLM only returned 1 series, it must be this one
    if len(vlm_results) == 1:
        only_val = next(iter(vlm_results.values()))
        only_key = next(iter(vlm_results.keys()))
        print(
            f"    [FALLBACK] Single-series match: '{series_name}' → '{only_key}'"
        )
        return only_val
    return {}


def _vlm_data_has_coverage(vlm_data: dict) -> bool:
    """Check if VLM data contains a coverage value."""
    cov = vlm_data.get("coverage_mid") or vlm_data.get("uptake_max")
    return cov is not None


# Aggregate isotherm performance data per material (coordinates)
if plot_mappings and plots:
    performance_data = aggregate_all_materials_performance(
        materials, plot_mappings, plots
    )
else:
    performance_data = {}

# ── Build VLM coverage lookup: material -> VLM coverage (via linking) ──
# IMPORTANT: A material may appear in multiple plots (e.g., different isotherms).
# We must NOT let a non-porous result from one plot overwrite a valid coverage from another.
# Strategy: only overwrite if the new data has a coverage value, or if no data exists yet.
vlm_cov_per_material = {}
for mapping in plot_mappings:
    plot_idx = mapping.plot_index
    if plot_idx not in cov_from_vlm:
        continue
    vlm_results_for_plot = cov_from_vlm[plot_idx]
    for sm in mapping.mappings:
        # Use fuzzy matching to find the VLM coverage entry
        vlm_data = _find_vlm_coverage_for_series(
            sm.series_name, vlm_results_for_plot
        )
        if vlm_data:
            existing = vlm_cov_per_material.get(sm.material_name)
            if existing is None:
                # No data yet — use whatever we have
                vlm_cov_per_material[sm.material_name] = vlm_data
            elif _vlm_data_has_coverage(vlm_data):
                if not _vlm_data_has_coverage(existing):
                    # New data has coverage, old data doesn't — upgrade
                    vlm_cov_per_material[sm.material_name] = vlm_data
                # else: both have coverage — keep the first one (from the earlier plot)

# ── Collect UNMATCHED series (materials in plots but not in materials list) ──
unmatched_vlm_cov = {}  # {series_name: {coverage data}}
for mapping in plot_mappings:
    plot_idx = mapping.plot_index
    if plot_idx not in cov_from_vlm:
        continue
    for series_name in mapping.unmatched_series:
        vlm_data = _find_vlm_coverage_for_series(
            series_name, cov_from_vlm[plot_idx]
        )
        if vlm_data:
            unmatched_vlm_cov[series_name] = vlm_data

# ── Fuzzy-match text coverage to materials ──
text_cov_per_material = {}
for material in materials:
    matches = _find_matching_text_coverage(material, cov_from_text)
    if matches:
        # Use the best match (highest coverage, porous preferred)
        best_key, best_entry = matches[0]
        text_cov_per_material[material] = best_entry
        # Also store ALL variants for the summary
        text_cov_per_material[material]["_all_variants"] = [
            {"condition": k, **v} for k, v in matches
        ]

# ── Summary Table ──
print("=" * 90)
print(
    f"{'Material':<40} {'Porous?':<7} {'cov_text':>12} {'cov_VLM':>12} {'cov_onset_VLM':>14} {'cov_zero_VLM':>14}"
)
print("-" * 90)


def _fmt(val, suffix=" m^2/g"):
    return f"{val:.1f}{suffix}" if val is not None else "NR"


for material in materials:
    text_entry = text_cov_per_material.get(material, {})
    text_porous = text_entry.get("porous", None)
    text_cov = text_entry.get("coverage")

    vlm_entry = vlm_cov_per_material.get(material, {})
    vlm_porous = vlm_entry.get("porous", None)
    vlm_cov = vlm_entry.get("coverage_mid") or vlm_entry.get("uptake_max")
    vlm_onset = vlm_entry.get("coverage_onset")
    vlm_zero = vlm_entry.get("coverage_zero")

    if text_porous is not None:
        porous_str = "YES" if text_porous else "NO"
    elif vlm_porous is not None:
        porous_str = "YES" if vlm_porous else "NO"
    else:
        porous_str = "?"

    print(
        f"{material:<40} {porous_str:<7} {_fmt(text_cov):>12} {_fmt(vlm_cov):>12} {_fmt(vlm_onset):>14} {_fmt(vlm_zero):>14}"
    )

    # Show all text coverage variants if multiple
    variants = text_entry.get("_all_variants", [])
    if len(variants) > 1:
        for v in variants:
            cond = v.get("condition", "?")
            vcov = v.get("coverage")
            print(f"  {'└ ' + cond:<38} {'':>7} {_fmt(vcov):>12}")

# Show unmatched series (in plot but NOT in materials list)
if unmatched_vlm_cov:
    print("-" * 90)
    print("UNMATCHED SERIES (in isotherm plot but not in materials list):")
    for series_name, vlm_data in unmatched_vlm_cov.items():
        porous = "YES" if vlm_data.get("porous") else "NO"
        print(
            f"  {series_name:<38} {porous:<7} {'':>12} {_fmt(vlm_data.get('coverage_mid')):>12} {_fmt(vlm_data.get('coverage_onset')):>14} {_fmt(vlm_data.get('coverage_zero')):>14}"
        )

print("=" * 90)
print(f"\nMaterials with isotherm data: {len(performance_data)}")
print(
    f"Materials with VLM coverage: {sum(1 for v in vlm_cov_per_material.values() if _vlm_data_has_coverage(v))}"
)
print(
    f"Materials with text coverage: {sum(1 for v in text_cov_per_material.values() if v.get('coverage') is not None)}"
)
if unmatched_vlm_cov:
    print(f"Unmatched series with coverage: {len(unmatched_vlm_cov)}")

---
## Step 10: Save Results

In [ ]:
from llm_synthesis.utils.performance_utils import sanitize_filename

paper_dir = os.path.join(OUTPUT_DIR, paper.id)
os.makedirs(paper_dir, exist_ok=True)

# Build and save per-material results
final_results = []

for entry in all_syntheses:
    mat = entry.material

    # Coverage from text (using fuzzy-matched lookup)
    text_cov_entry = text_cov_per_material.get(mat, {})
    # Remove internal helper key before saving
    text_cov_clean = {
        k: v for k, v in text_cov_entry.items() if not k.startswith("_")
    }

    # Coverage from VLM
    vlm_cov_entry = vlm_cov_per_material.get(mat, {})

    result = {
        "material": mat,
        "synthesis": entry.synthesis.model_dump() if entry.synthesis else None,
        "evaluation": entry.evaluation.model_dump()
        if entry.evaluation
        else None,
        "tc_from_text": text_cov_clean if text_cov_clean else None,
        "tc_from_text_all_variants": [
            {k: v for k, v in variant.items()}
            for variant in text_cov_entry.get("_all_variants", [])
        ]
        or None,
        "tc_from_vlm": {
            "superconducting": vlm_cov_entry.get("superconducting"),
            "T_onset": vlm_cov_entry.get("t_onset"),
            "Tc_mid": vlm_cov_entry.get("tc_mid"),
            "T_zero": vlm_cov_entry.get("t_zero"),
            "Delta_Tc": vlm_cov_entry.get("delta_tc"),
        }
        if vlm_cov_entry
        else None,
        "performance": (
            performance_data[mat].model_dump()
            if mat in performance_data
            else None
        ),
    }
    final_results.append(result)

    # Save individual material file
    mat_name = sanitize_filename(mat)
    mat_path = os.path.join(paper_dir, f"{mat_name}.json")
    with open(mat_path, "w") as f:
        json.dump(result, f, indent=2, default=str)

# Save plot mappings
if plot_mappings:
    with open(os.path.join(paper_dir, "performance_mappings.json"), "w") as f:
        json.dump([m.model_dump() for m in plot_mappings], f, indent=2)

# Save Tc summary (includes unmatched series)
tc_summary = {
    "paper_id": paper.id,
    "materials": materials,
    "cov_from_text_raw": cov_from_text,
    "cov_from_text_matched": {
        k: {kk: vv for kk, vv in v.items() if not kk.startswith("_")}
        for k, v in text_cov_per_material.items()
    },
    "tc_from_vlm_per_material": {
        k: {kk: vv for kk, vv in v.items() if not isinstance(vv, list | dict)}
        for k, v in vlm_cov_per_material.items()
    },
    "tc_from_vlm_unmatched": {
        k: {kk: vv for kk, vv in v.items() if not isinstance(vv, list | dict)}
        for k, v in unmatched_vlm_cov.items()
    },
    "tc_from_vlm_raw": {str(k): v for k, v in cov_from_vlm.items()},
}
with open(os.path.join(paper_dir, "cov_summary.json"), "w") as f:
    json.dump(tc_summary, f, indent=2, default=str)

# Save overall summary
summary = {
    "paper_id": paper.id,
    "paper_name": paper.name,
    "total_materials": len(materials),
    "materials_list": materials,
    "total_plots_extracted": len(plots) if not SKIP_FIGURES else 0,
    "cov_p_plots_found": len(relevant_plots) if not SKIP_FIGURES else 0,
    "materials_with_text_tc": sum(
        1 for v in text_cov_per_material.values() if v.get("Tc_mid") is not None
    ),
    "materials_with_vlm_cov": len(vlm_cov_per_material),
    "unmatched_series_with_cov": len(unmatched_vlm_cov),
    "materials_with_cov_p_data": len(performance_data),
}
with open(os.path.join(paper_dir, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(f"[OK] Results saved to: {paper_dir}/")
print(f"   - {len(final_results)} material files")
print("   - cov_summary.json")
print("   - summary.json")
if plot_mappings:
    print("   - performance_mappings.json")

---
## Step 11: Export Flat Records + Append to Master CSV

Build one standardized record per material (flat, CSV-friendly) for multi-paper
aggregation. Each run appends to `MASTER_CSV` — duplicates are handled by the
`(paper_id, material)` composite key.

In [ ]:
import csv
from pathlib import Path

# ── Helpers ──


def extract_year_from_arxiv_id(paper_id: str) -> int | None:
    """Extract publication year from arXiv ID.

    Formats:
      YYMM.NNNNN  → 20YY  (e.g., 0708.3882 → 2007)
      YYMM.NNNNNvN → 20YY  (strip version suffix)
      Anything else → None
    """
    # Strip version suffix (e.g., "0708.3882v2" → "0708.3882")
    # Also handle compound IDs like "0708.3882_0708.3882v2"
    clean = paper_id.split("_")[0]  # take first part if compound
    clean = re.sub(r"v\d+$", "", clean)

    match = re.match(r"^(\d{2})(\d{2})\.\d+$", clean)
    if match:
        yy = int(match.group(1))
        # arXiv started in 1991; IDs like 07xx are 2007, 91xx would be 1991
        year = 2000 + yy if yy < 90 else 1900 + yy
        return year
    return None


def normalize_formula_for_csv(s: str) -> str:
    """Normalize a material formula for cross-paper deduplication.

    Strips unicode subscripts → ASCII digits, δ→delta, removes spaces,
    strips parenthetical condition suffixes like '(9hr-reduced)'.
    Preserves case (unlike _normalize_formula which lowercases).
    """
    # Strip parenthetical condition suffixes
    base = re.sub(r"\s*\([^)]*\)\s*$", "", s).strip()
    # Unicode subscripts → ASCII
    sub_map = {
        "₀": "0",
        "₁": "1",
        "₂": "2",
        "₃": "3",
        "₄": "4",
        "₅": "5",
        "₆": "6",
        "₇": "7",
        "₈": "8",
        "₉": "9",
        "₋": "-",
        "₊": "+",
        "₍": "(",
        "₎": ")",
    }
    for uni, asc in sub_map.items():
        base = base.replace(uni, asc)
    # Greek letters
    base = base.replace("δ", "delta").replace("Δ", "Delta")
    # Normalize dashes
    base = base.replace("−", "-").replace("–", "-")
    # Remove stray spaces inside formula
    base = base.strip()
    return base


def pick_best_tc(
    text_tc: float | None, vlm_tc: float | None, text_onset: float | None
) -> tuple[float | None, str]:
    """Pick the best Tc value. Priority: text Tc_mid > text T_onset > VLM Tc_mid.

    Returns (tc_value, source_label).
    """
    if text_tc is not None:
        return text_tc, "text"
    if text_onset is not None:
        return text_onset, "text_onset"
    if vlm_tc is not None:
        return vlm_tc, "vlm"
    return None, "none"


# ── Build flat records ──

COLUMNS = [
    "paper_id",
    "year",
    "material",
    "material_normalized",
    "is_porous",
    "coverage_text",
    "coverage_text_onset",
    "coverage_text_zero",
    "coverage_text_source",
    "coverage_vlm",
    "coverage_vlm_onset",
    "coverage_vlm_zero",
    "coverage_vlm_source",
    "coverage_vlm_source_plot",
    "coverage_best",
    "coverage_best_source",
    "has_text_coverage",
    "has_vlm_coverage",
    "synthesis_method",
    "synthesis_score",
]

year = extract_year_from_arxiv_id(paper.id)
flat_records = []

for entry in all_syntheses:
    mat = entry.material

    # Text coverage
    text_entry = text_cov_per_material.get(mat, {})
    text_cov = text_entry.get("coverage")
    text_onset = text_entry.get("coverage_onset")
    text_zero = text_entry.get("coverage_zero")
    text_porous = text_entry.get("porous")
    text_source = None
    variants = text_entry.get("_all_variants", [])
    if variants:
        text_source = variants[0].get("condition")

    # VLM coverage
    vlm_entry = vlm_cov_per_material.get(mat, {})
    vlm_cov = vlm_entry.get("coverage_mid") or vlm_entry.get("uptake_max")
    vlm_onset = vlm_entry.get("coverage_onset")
    vlm_zero = vlm_entry.get("coverage_zero")
    vlm_porous = vlm_entry.get("porous")
    vlm_source = vlm_entry.get("source", "main plot") if vlm_entry else None

    # Find which figure the VLM coverage came from
    vlm_source_plot = None
    for mapping in plot_mappings:
        for sm in mapping.mappings:
            if sm.material_name == mat:
                vlm_source_plot = mapping.figure_reference
                break
        if vlm_source_plot:
            break

    # Is porous?
    if text_porous is not None:
        is_porous = text_porous
    elif vlm_porous is not None:
        is_porous = vlm_porous
    else:
        is_porous = None  # unknown

    # Best coverage
    coverage_best, coverage_best_source = pick_best_tc(
        text_cov, vlm_cov, text_onset
    )

    # Synthesis info
    synth_method = entry.synthesis.synthesis_method if entry.synthesis else None
    synth_score = (
        entry.evaluation.scores.overall_score
        if entry.evaluation and entry.evaluation.scores
        else None
    )

    record = {
        "paper_id": paper.id,
        "year": year,
        "material": mat,
        "material_normalized": normalize_formula_for_csv(mat),
        "is_porous": is_porous,
        "coverage_text": text_cov,
        "coverage_text_onset": text_onset,
        "coverage_text_zero": text_zero,
        "coverage_text_source": text_source,
        "coverage_vlm": vlm_cov,
        "coverage_vlm_onset": vlm_onset,
        "coverage_vlm_zero": vlm_zero,
        "coverage_vlm_source": vlm_source,
        "coverage_vlm_source_plot": vlm_source_plot,
        "coverage_best": coverage_best,
        "coverage_best_source": coverage_best_source,
        "has_text_coverage": text_cov is not None,
        "has_vlm_coverage": vlm_cov is not None,
        "synthesis_method": synth_method,
        "synthesis_score": synth_score,
    }
    flat_records.append(record)

# ── Save per-paper JSONL (one line per material, flat) ──
jsonl_path = os.path.join(paper_dir, "coverage_flat_records.jsonl")
with open(jsonl_path, "w") as f:
    for rec in flat_records:
        f.write(json.dumps(rec, default=str) + "\n")
print(f"[OK] Saved {len(flat_records)} flat records → {jsonl_path}")

# ── Append to master CSV (create with header if new, skip duplicates) ──
master_path = Path(MASTER_CSV)
master_path.parent.mkdir(parents=True, exist_ok=True)

# Read existing rows to detect duplicates by (paper_id, material)
existing_keys = set()
if master_path.exists():
    with open(master_path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            existing_keys.add(
                (row.get("paper_id", ""), row.get("material", ""))
            )

# Determine which records are new
new_records = [
    r
    for r in flat_records
    if (r["paper_id"], r["material"]) not in existing_keys
]
replaced_records = [
    r for r in flat_records if (r["paper_id"], r["material"]) in existing_keys
]

if replaced_records:
    # Re-run of same paper: replace old rows with new ones
    # Read all existing rows, filter out the ones we're replacing
    replace_keys = {(r["paper_id"], r["material"]) for r in replaced_records}
    all_rows = []
    if master_path.exists():
        with open(master_path, newline="") as f:
            reader = csv.DictReader(f)
            all_rows = [
                row
                for row in reader
                if (row.get("paper_id", ""), row.get("material", ""))
                not in replace_keys
            ]
    # Add all current records (both new and replaced)
    all_rows.extend(
        {k: (str(v) if v is not None else "") for k, v in r.items()}
        for r in flat_records
    )
    # Rewrite entire file
    with open(master_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=COLUMNS)
        writer.writeheader()
        writer.writerows(all_rows)
    print(
        f"[OK] Master CSV updated (replaced {len(replaced_records)} existing + "
        f"added {len(new_records)} new) → {master_path}"
    )
else:
    # All new: just append
    write_header = not master_path.exists() or master_path.stat().st_size == 0
    with open(master_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=COLUMNS)
        if write_header:
            writer.writeheader()
        for rec in flat_records:
            writer.writerow(
                {k: (str(v) if v is not None else "") for k, v in rec.items()}
            )
    print(
        f"[OK] Appended {len(new_records)} records to master CSV → {master_path}"
    )

# ── Display flat records ──
print(f"\n{'=' * 100}")
print(f"FLAT RECORDS FOR DOWNSTREAM (paper_id={paper.id}, year={year})")
print("=" * 100)
print(
    f"{'Material':<35} {'Porous?':<7} {'cov_text':>10} {'cov_VLM':>10} {'cov_best':>10} {'Source':<12} {'Synth':<15}"
)
print("-" * 100)
for rec in flat_records:
    porous = (
        "YES"
        if rec["is_porous"]
        else ("NO" if rec["is_porous"] is False else "?")
    )
    cov_t = f"{rec['coverage_text']:.1f}" if rec["coverage_text"] else "—"
    cov_v = f"{rec['coverage_vlm']:.1f}" if rec["coverage_vlm"] else "—"
    cov_b = f"{rec['coverage_best']:.1f}" if rec["coverage_best"] else "—"
    src = rec["coverage_best_source"]
    synth = rec["synthesis_method"] or "—"
    print(
        f"{rec['material']:<35} {porous:<7} {cov_t:>10} {cov_v:>10} {cov_b:>10} {src:<12} {synth:<15}"
    )

---

## Done!

Results include for each material:
- **Synthesis procedure** (GeneralSynthesisOntology)
- **material_coverage (cov) from text** (values explicitly reported in the paper)
- **material_coverage from VLM**
- **cov(p) performance data** (full (cov, p) coordinates linked to materials)